In [ ]:
!pip install -q fastapi uvicorn pyngrok "diffusers[torch]" accelerate safetensors pillow python-multipart


In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pyngrok import ngrok
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import torch, tempfile, os, threading, uvicorn


app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# choose device
device = "cuda" if torch.cuda.is_available() else "cpu"

# GPU‑friendly Stable Diffusion v1.5 img2img
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True,
).to(device)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
@app.post("/api/img2img")
async def img2img(
    prompt: str = Form(...),
    strength: float = Form(0.5),
    guidance_scale: float = Form(7.5),
    image: UploadFile = File(...)
):
    try:
        init_image = Image.open(image.file).convert("RGB")
        init_image = init_image.resize((512, 512))

        result = pipe(
            prompt=prompt,
            image=init_image,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=25,
        ).images[0]
    except Exception as e:
        print("IMG2IMG ERROR:", repr(e))   # <-- check Colab output
        raise

    tmp_dir = tempfile.mkdtemp()
    out_path = os.path.join(tmp_dir, "img2img.png")
    result.save(out_path)
    return FileResponse(out_path, media_type="image/png", filename="img2img.png")


In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=8002)

thread = threading.Thread(target=run, daemon=True)
thread.start()

ngrok.set_auth_token("37LTC0pnKHQw0M9oGfHSjVaKO65_7PosfHwhrGYxbV9m8YTqz")
public_url_img2img = ngrok.connect(8002, "http")
public_url_img2img


INFO:     Started server process [472]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8002 (Press CTRL+C to quit)


<NgrokTunnel: "https://prescriptively-hypertrophic-daren.ngrok-free.dev" -> "http://localhost:8002">